# SmartPixels Tier-2

## From inner-pixel hits to Level-1 trigger capability

**digiRefit &rarr; refit-quality BDT &rarr; coherent downstream reco &rarr; jet tagging & vertexing**

Nick Manganelli &mdash; SmartPixels internal talk &mdash; July 2026

<small>Interactive deck: the refit-replay visualizer and the MVA explorer are live HTML &mdash; links on their slides.</small>

**[0:00 &ndash; 0:30]** Welcome. This is an internal status talk: everything shown exists and runs today.
Audience calibration: we have CMS/ATLAS experts, Muon Collider students, undergrads, and ASIC
engineers in the room &mdash; everyone knows pixel detectors; not everyone knows CMS L1 jargon, so
I'll define terms as they appear. 30 minutes of content, dark-sector physics in backup.

## The thesis

> **Better inner-pixel information &rarr; better L1 tracks &rarr; new trigger capability.**

CMS Phase-2 L1 tracks are built from the **outer tracker only** (r &gt; 25 cm).
SmartPixels asks: what if the trigger could also see the **inner pixel barrel** (r &lt; 16 cm),
where impact parameters and vertices actually live?

**The R&D-payoff framing** &mdash; each layer of this work stands on its own:

| layer | Phase-2-baseline value *even if SmartPixels is never adopted* |
|---|---|
| track refit machinery | pixel-augmented L1 tracking studies, truth-linked nano tooling |
| refit-quality BDT | track-quality MVA methodology, bit-exact FPGA (conifer) path |
| tagger / vertex R&D | L1 jet-flavor + vertexing improvements on the baseline track trigger |

**[0:30 &ndash; 1:30]** One minute. The thesis sentence is the whole talk. For non-CMS people: the
Level-1 trigger is the 40 MHz hardware decision layer; Phase-2 CMS puts *tracking* in it for the
first time, but only from the outer tracker. The payoff table is the de-risking argument: this
R&D produces upstreamable spinoffs regardless of whether SmartPixels itself is adopted &mdash;
say this explicitly, it frames every result that follows.

### Cheat sheet (CMS-isms used in this talk)

| term | meaning |
|---|---|
| **L1 / L1T** | Level-1 trigger: 40 MHz custom-hardware decision layer (FPGA, &micro;s latency) |
| **OT / IT** | outer tracker (r &asymp; 25&ndash;110 cm) / inner tracker = pixels (TBPX barrel, r &asymp; 3&ndash;16 cm) |
| **L1TT / TTTrack** | Phase-2 L1 Track Trigger; its track object (r<sub>inv</sub>, &phi;<sub>0</sub>, tan&lambda;, z<sub>0</sub>, d<sub>0</sub>) |
| **stub** | correlated hit pair in an OT p<sub>T</sub>-module &mdash; the L1 tracking primitive |
| **TP** | TrackingParticle: the truth particle a reco track is matched to |
| **GTT** | Global Track Trigger: track-only trigger objects (primary vertex, sums) |
| **PF / PUPPI** | Particle Flow (link tracks+calo into candidates) / per-particle pileup weighting |
| **SC4 / SC8** | seeded-cone L1 jets, R = 0.4 / 0.8 |
| **KF** | Kalman filter: sequential fit update, one measurement at a time |

Skippable cheat sheet &mdash; keep it up for 15 seconds, tell people to photograph it. For the
undergrads: the one concept to hold is that a *stub* is the outer-tracker primitive and a
*TTTrack* is a 5-parameter helix fit through stubs, with the two impact parameters d0 (transverse)
and z0 (longitudinal) describing where the track passes closest to the beamline.

## Where Tier-2 came from: the tier model

Separate **what exists** from **what is being rehearsed**:

| tier | status | what it is |
|---|---|---|
| **Tier 3 &mdash; `refit`** | reserved | the true system: consumes a real SmartTracklet collection from a future L1 SmartPixels track finder (mode name + config plumbed, raises if selected) |
| **Tier 2 &mdash; `digiRefit`** | **implemented** | OT L1Tracks projected into TBPX; hit candidates are the **real persisted pixel digis** &mdash; occupancy, combinatorics and fakes come for free; only the cluster **angle** information is synthesized |
| Tier 1 &mdash; TP-projection toys | footnote | degenerate path of the same producer |

Tier-2 truth is *per-digi* via `PixelDigiSimLink`: a same-TP digi gets angles from the track's own
parent, an other-TP digi from **its** parent, a no-link digi from a noise distribution &mdash;
**correct fake phenomenology by construction**.

**[1:30 &ndash; 3:30]** Two minutes, one slide. The point for experts: Tier 2 is not a toy &mdash; the
digis are the real ones from the simulation chain, so window occupancy and combinatoric fakes are
honest; the *only* synthesized ingredient is the cluster-shape angle info a SmartPixels ASIC would
provide (that is what the ASIC engineers in the room would eventually supply for real). One
truth-posture line if asked: production runs "posture C" &mdash; tracks rebuilt in-job from the
file's persisted stubs, giving real pileup *and* genuine seed covariance. Details in backup.

## NanoAOD capability, today

One flat, columnar file per configuration carries the **whole loop** (production &rarr; training &rarr; eval):

- **Extended SmartPix nano flavors** &mdash; tracks + PF + jets + gen in one file (`L1PFTrkNanoSmartPix+withGen`)
- **Per-variant track tables** &mdash; refit output for each layer config, row-aligned with the reference `L1TTrack`
- **Refit sidecar** &mdash; per-hit link tables: &chi;&sup2; increments, pulls, window multiplicity, hit-truth class
- **Truth by index** &mdash; matched-TP *full* parameters on every track (`tp_d0`, `tp_z0`, `tp_phi`, production vertex) &rarr; resolution-vs-truth offline, no re-matching
- **Jet side** &mdash; SC4 **and** SC8 seeded-cone jets, PF-candidate link tables (&minus;1/&minus;2 sentinel convention), embedded OT stubs

**[3:30 &ndash; 6:30]** Three minutes over two slides. Skip the history entirely &mdash; this is "what we
CAN do now". For the tree-level-analysis crowd: this is NanoAOD, so everything downstream is
uproot/awkward, no framework needed. The key design idea to voice: *row alignment* &mdash; variant
tables are index-aligned to the reference track table, so truth and features cross-reference by
index, never by geometric re-matching.

### Two coherent nano products (both proven in production)

**Fat coherent nano** &mdash; *one* config, *fully coherent downstream*:
refit tracks &rarr; **re-run Layer-1 correlator + vertex in-job** &rarr; PF &rarr; PUPPI &rarr; SC4/SC8 jets
all computed **from the refit tracks** (`nano_fat_{1111,1100}_coopt`, `0000_baseline`).

- Proven non-trivial: naive downstream injection was byte-identical to the file's products; the
  in-job correlator re-run is what makes jets/vertex/PF actually *see* the refit.

**Multi-mode co-registered nano** &mdash; *all 15 refit variants per event*, one coherent shard
(`nano_pG`): every layer config side by side on the same events &rarr; config ablations from a single file.

The subtle point (worth 30 seconds for the experts): the correlator inputs are persisted
products built from the *original* tracks, so simply relabeling collections changes nothing &mdash;
we proved a naive injection produced byte-identical jets. The fix re-emulates the correlator
regions + the FastHisto vertex inside the nano job, pointed at the refit tracks. That is what
"coherent" means every time I say it in the tagger section.

# Tier-2 `digiRefit`

## the approach, what works, and what does not (yet)

**[6:30]** Section marker &mdash; announce this is the core ~11 minutes of the talk.

## The idea

![digiRefit idea](figures/digirefit_idea.png)

**5-par OT-only seed vs 5-par OT+IT refit** (`promptHnpar=5`, real fitted d<sub>0</sub>):
project the TTTrack helix into TBPX, open a **per-layer window of real pixel digis**,
synthesize the cluster angles (&alpha;, &beta;), select at most one hit per layer,
and apply a **Kalman update** to the 5-parameter track state at each layer.

Producer pipeline (per track): *window collection &rarr; digi truth-classification & angle synthesis
&rarr; hit selection &rarr; sequential scalar KF update (numerical Jacobian) &rarr; loud-failure guards.*
Same helix projector shared with the payload analyzer &mdash; single implementation, no drift.

**[6:30 &ndash; 8:30]** The figure is the explanation: left panel shows the lever arm (track fit way
out at 25&ndash;110 cm, extrapolated inward), right panel the per-layer windows and the POCA where
d0/z0 sharpen. For undergrads: a Kalman update is just "merge the prediction with one new
measurement, weighted by uncertainties". For experts: 5-parameter *prompt* seed &mdash; we
deliberately anchor on promptHnpar=5 so the seed has a real fitted d0; the b-tagging /di-Higgs
use-case demands it, and the 4-par pinned-d0 path is dropped everywhere in this talk.

## The centerpiece: step-by-step refit replay <span style="font-size:60%">(interactive)</span>

**Open in a browser tab:** [`../eval_refitq/refitviz/refit_replay.html`](../eval_refitq/refitviz/refit_replay.html)
&mdash; self-contained ~17 MB HTML, no server, no kernel.

<iframe src="../eval_refitq/refitviz/refit_replay.html" width="100%" height="600" style="border:1px solid #999;"></iframe>

- **per-KF-step replay**: slider / play / step buttons through each layer's update
- seed (OT-only) vs refit helix, **covariance bands**, per-step parameter deltas
- **full-hit-set &chi;&sup2; column** &mdash; reduced &chi;&sup2; *including the OT stubs*, seed row &rarr; per-layer rows &rarr; refit row &rarr; truth footer
- **15 layer configs &times; 3 angle modes** (none / &alpha; / &alpha;+&beta;), curated real tracks (clean genuine / wrong-hit pickup / fake)

**[8:30 &ndash; 12:00]** THE demo &mdash; give it 3+ minutes. Have it open in a dedicated browser tab
beforehand (the iframe works in the exported slides; in JupyterLab prefer the tab). Suggested
script: (1) clean genuine track, AAAA, alphaBeta &mdash; step the KF layer by layer, watch d0/z0
bands tighten toward the truth row; (2) flip angle mode none&rarr;alphaBeta on the same track to
show what the ASIC angle info buys; (3) the fake track &mdash; watch the full-hit-set chi2 column
explode: that is a *feature*, it is the discrimination the quality BDT will use. For ASIC
engineers: the angle mode toggle is exactly their deliverable's physics value, live.

## The sign bug: what the visualizer caught

**Act 1 &mdash; the symptom.** Long-running L3/L4 inefficiency and KF instability;
&chi;&sup2; blow-ups; a saga of clamp "band-aids". Aggregate metrics: nothing obviously wrong.

**Act 2 &mdash; the visualizer.** First event on screen: the projected seed helix curled to the
**mirror-image azimuthal side** of its own hits &mdash; the helix projector's parametrization curled
*opposite* to the TTTrack r<sub>inv</sub> sign convention. Error grows as **r&sup2;/R**: &asymp;0 at L1,
**&asymp;2 mm at L4** &rarr; pulls up to **142&sigma;**. Invisible at L1, fatal at L4.

**Act 3 &mdash; triple verification, one-line fix.**
1. circle-fit through the track's own OT stubs: R = 388 cm = stored 1/|r<sub>inv</sub>| &mdash; helix lands on its stubs only with **negated** curvature
2. per-layer residuals grow monotonically L1&rarr;L4 (0.23 mm &rarr; 2.3 mm); post-fix flat at &asymp;0.05 mm
3. matched-TP truth helix confirms the stored curvature sign was right &mdash; **the formula was wrong**

Fix: `const double rInv = -h.rInv;` at projector entry. The clamps it was fighting are now inert.

> **Lesson: interactive event-level inspection catches what aggregate metrics hide.**

**[12:00 &ndash; 14:30]** The narrative beat &mdash; tell it as a story, ~2.5 min. Emphasize the shape of
the failure: a *sign* error whose magnitude grows quadratically with radius, so every L1-anchored
sanity check passed. The triple verification matters for credibility with the experts: geometry
(stubs), statistics (residual growth), truth (TP handedness) &mdash; three independent witnesses.
And the punchline for everyone: months of clamp archaeology, one line of code.

## Post-fix: truth-validated resolution at scale

![resolution](figures/resolution_d0z0.png)

**184,839 matched tracks / 1000 PU events** (10-file post-fix production, matched-TP truth):
**d<sub>0</sub> 389 &rarr; 210 &micro;m (&minus;46%)**, **z<sub>0</sub> 1430 &rarr; 626 &micro;m (&minus;56%)** for the all-layer config.
The 5-par prompt seed (`promptHnpar=5`) is validated at production scale.

Note the fine print the medians reveal: **AAII (L1+L2 only) beats AAAA on the median** &mdash;
outer IT layers add wrong-hit pickup (multiple scattering widens the window) alongside their
constraint. Real, understood, and part of the limitations slide.

**[14:30 &ndash; 16:00]** The money plot for the refit. Everything here is |reco &minus; matched-TP truth|
median, not seed&rarr;refit deltas &mdash; resolution *toward truth* is the honest metric.
For the b-tagging crowd: 210 &micro;m median d0 at L1, from a trigger-track refit, is the quantity
that feeds impact-parameter-based flavor tagging. The AAII-vs-AAAA inversion is a gift: it
pre-announces the wrong-hit-pickup limitation with data, not apology.

## Limitations, stated plainly

- **Low statistics**: ~1000 events per config; every downstream number inherits this
- **Fake tracks**: the refit &chi;&sup2; explodes on fakes &mdash; *by design a feature*
  (the full-hit-set &chi;&sup2; column is the discriminant), but fake-side behavior is measured on few-thousand-track samples
- **MS-driven wrong-hit pickup on outer IT layers** &mdash; visible in the AAII &lt; AAAA medians; needs smarter per-layer windows / hit arbitration
- **ASIC efficiency payloads are placeholders** &mdash; the hooks exist (`smarthitTrueSet` reserved), but no real ASIC response model is wired in yet
- **KF angle clamps under review**: default `measAngleMaxAbs=12` is tighter than needed post-fix
  (fires on 0.12% of accepted-hit angles); relax to ~30&ndash;50. The clamp now only guards the
  *persisted &chi;&sup2; feature* from grazing-crossing garbage &mdash; it is **resolution-neutral** (clamp on/off identical to the last digit)

**[16:00 &ndash; 17:30]** Do not rush this slide; its honesty buys the tagger claims credibility.
The clamp story in one breath: the sign bug made the clamps look load-bearing; post-fix they are
inert for resolution and only keep the persisted chi2 BDT-input column clean &mdash; the coopt
production already runs measAngleMaxAbs=30. If ASIC engineers ask: the efficiency-payload hook is
exactly where their measured response model plugs in.

# New capabilities on top of the refit

## quality BDT &rarr; coherent tagger matrix &rarr; vertex R&D

**[17:30]** Section marker: ~7 minutes. The ladder: per-track quality score, then the
event-level question &mdash; does any of this help *jets*.

## Refit-quality BDT (&ldquo;tkquality&rdquo;)

![stage3](figures/stage3_auc_configs.png)

- **24-feature v1** spec (per-layer pulls, &chi;&sup2; increments, window occupancy, seed deltas, OT fit words)
- genuine-vs-fake **AUC 0.970&ndash;0.977 across ALL 15 layer configs** &mdash; robust to which layers are instrumented
- **conifer &harr; xgboost bit-parity**: offline float32 tree-walk matches the in-producer `conifer::BDT`
  (margin self-check exactly 0.0; producer parity 16410/16455 bit-exact, residuals = float32 snapshot threshold-edge cases)

**[17:30 &ndash; 19:30]** Two minutes. Orienting sentence: this is a per-track "is it real?" score,
trained genuine-vs-fake on the refit sidecar features. The flat 0.970&ndash;0.977 band is the
headline: instrument any subset of layers and the quality score works &mdash; a robustness statement
the hardware people should hear. Bit-parity matters because conifer is the FPGA path: what we
train offline is bit-for-bit what the firmware would compute.

## Does it help the jet tagger?

![stage4](figures/stage4_auc_matrix.png)

**Headline: refit-driven *coherent* downstream reco lifts the 8-flavor jet-tagger macro AUC
by +0.044 (1111) / +0.027 (1100) over the OT-only baseline (0.686 &rarr; 0.730 / 0.713)** &mdash;
well outside the seed spread (&plusmn;0.002&ndash;0.014).

**Caveat, same breath:** ~1300 test jets per view &mdash; *only* the 0.03&ndash;0.04 refit-vs-OT-only
gap is robust; every within-view feature delta is at or below seed noise.

**[19:30 &ndash; 21:30]** The tagger headline and its caveat live on the same slide on purpose.
Mechanism, one sentence: the lift is NOT from adding a feature to the tagger &mdash; it comes from
the vertex&rarr;PF&rarr;PUPPI&rarr;jet chain being *re-computed from the refit tracks* (the fat
coherent nanos), i.e., better tracks propagate through the whole event reconstruction. For
skeptics: 3 seeds per cell, best-of-seed shown with spread; the 11-cell matrix is in the json.

## The feature-level story is coherent too

- **refit-BDT score as a tagger input**: small positive (+0.003, within noise) &mdash; but consistently
  **lifts taus** (&tau;<sup>+</sup> AUC 0.706 &rarr; 0.733 on 1111)
- **vertex-d<sub>xy</sub> feature helps only where there is no refit**: 0000 baseline 0.686 &rarr; 0.694
  (+0.008); redundant on refit views &mdash; *once tracks carry a real impact parameter, the vertex proxy adds nothing*
- **charge head trains for free**: +charge output on the full feature set, macro AUC 0.722 &mdash; no tagging cost

Each feature behaves exactly as the physics says it should &mdash; that self-consistency is the
evidence the machinery is measuring something real (at these stats).

**[21:30 &ndash; 22:30]** One minute. This is the "physics sanity" slide: at ~1300 test jets no
within-view delta is significant, but the *pattern* &mdash; vertex-dxy only helps the view without
refit IP, the quality score helps the multiprong taus &mdash; is what you predict before looking.
Say that sentence; it is the strongest low-stats statement one can honestly make.

## Second interactive: the MVA explorer

**Serve it** (binary model files &mdash; `file://` will not fetch them):
```
cd ngtagger-train && python3 -m http.server 8000
```
then open [`http://localhost:8000/eval_mva_explorer/site/explorer.html`](http://localhost:8000/eval_mva_explorer/site/explorer.html)

<iframe src="http://localhost:8000/eval_mva_explorer/site/explorer.html" width="100%" height="560" style="border:1px solid #999;"></iframe>

- panels for **regressions / tkquality / tagger**; overlay any tuple of trainings
- working-point efficiency curves, per-bin AUC, score distributions &mdash; all client-side

**[22:30 &ndash; 24:30]** Second demo, ~2 minutes; shorter than the refit replay. Show one overlay:
tkquality score distribution genuine-vs-fake, then the tagger per-bin AUC. If time is tight this
demo is the first thing to compress &mdash; the deck's screenshots carry the point. Remember to
start the http.server BEFORE the talk (it serves the .bin model files the page fetches).

## Vertex emulators (one slide)

Context: baseline GTT primary vertex is **FastHisto** (histogram tracks in z<sub>0</sub>, pick the peak);
**NNVtx** is the ML alternative. Our R&D on top of the refit tracks:

- **transverse vertex (d<sub>x</sub>, d<sub>y</sub>) least-squares**: closed-form normal equations from
  track (d<sub>0</sub>, &phi;<sub>0</sub>) &mdash; a beam-spot-scale transverse fit from L1 tracks, with significance
- **kernel peak-finder**: replace FastHisto's flat window with tapered kernels
  (triangular / Gaussian / Epanechnikov) &mdash; in two-close-vertices toys the tapered kernel
  **halves the midpoint mis-pick rate**
- feeds the tagger's `vertexdxy` feature group (the one that lifted the OT-only view)

**[24:30 &ndash; 26:00]** ~1.5 minutes. Orienting sentence: the L1 primary vertex today is a 1-D
histogram peak in z; we prototyped (a) the *transverse* companion fit that only becomes meaningful
once tracks have a real d0 &mdash; i.e., only with the refit &mdash; and (b) a drop-in kernel upgrade to
the histogram peak-finder itself, which is a pure Phase-2-baseline improvement (payoff framing again).

## Close: the payoff, and what's next

**Every rung already paid out**: refit machinery (truth-linked nano loop) &rarr; quality BDT
(bit-exact FPGA path, robust across configs) &rarr; coherent reco (+0.04 tagger AUC at low stats)
&mdash; each with Phase-2-baseline value independent of SmartPixels adoption.

**Next:**
1. **Real production statistics** &mdash; the recipe is productionized; waiting on 200-PU input procurement
2. **ASIC-fidelity payloads** &mdash; wire measured ASIC response/efficiency into the reserved hooks
3. **The TS0 lever &mdash; score transmission**: today the quality-BDT score reaches the tagger *only offline*.
   Making it visible to PF track selection is a **config-only** change (cut-strings can read `trkMVA1`);
   carrying it into the **candidate word** for the tagger is a hardware/word-format change &mdash;
   the concrete, costed ask for the hardware discussion.

**[26:00 &ndash; 27:00]** Close on the thesis. The TS0 lever is the actionable takeaway for the ASIC
and firmware people: there is a zero-hardware step (config-only PF visibility) and a
word-format step, cleanly separated. Then offer backup: dark-sector reach if there is time.
Leave ~3 minutes for questions inside the 30.

# Backup

*dark-sector reach &middot; truth postures &middot; KF guards &middot; per-flavor tables &middot; parity details*

Backup divider. The dark-sector slide is the designated if-time bonus.

## Backup: dark-sector L1 sensitivity (the bonus physics)

- Signature map: soft-unclustered energy patterns (**SUEP** flagship &mdash; rebuilt after the
  multiplicity-based version was refuted), displaced/low-p<sub>T</sub> track signatures needing IT-level info at L1
- **One placement, two capabilities**: a proposed subsystem between
  {L1TrackFinder + SmartPixels track finder} and {GTT / GMT / correlator} &mdash;
  graph-track **condensation** + **reseeding**
- **Upstream-payoff thesis**: the observables and the condensation/reseeding subsystem are
  contributions to the *baseline Phase-2 track trigger*, decoupled from SmartPixels adoption risk
  &mdash; speculative R&D de-risked by its spinoffs

If-time bonus (~2 min if used). Scoping + critique docs exist
(DarkSectorL1Scoping.md); frame as reach, not results.

## Backup: truth postures (why three)

Association maps are Ref-keyed &mdash; mixing collections silently breaks truth. Hence:

- **Posture B `inJob`** (default): remake digis&rarr;stubs&rarr;tracks&rarr;maps self-consistently &mdash; all noPU development
- **Posture A `fromFile`**: file tracks + file maps + file digis &mdash; needed when re-digitizing would lose PU;
  but old-layout file tracks have all-zero helix covariance &rarr; forces parametrized seeding
- **Posture C `fromFileStubs`** (production): rebuild tracks from the file's persisted **stub** tier &rarr;
  real PU **and** genuine seed covariance (`seedCovMode="trackCov"` valid on PU files)

## Backup: KF guards &mdash; post-fix status

| guard | default | post-fix firing | verdict |
|---|---|---|---|
| `jacobianMaxAbs` | 10&#8308; | **0** | inert (was a bug band-aid); can drop |
| `chi2UpdateGate` | 2&times;10&#8310; | **0** | keep as cheap numerical backstop |
| `predAngleMaxAbs` | 12 | ~5 / 100 evt | cosmetic; relax freely |
| `measAngleMaxAbs` | 12 | 0.12% of accepted-hit angles | **relax 12 &rarr; ~30&ndash;50**, keep as guard |

Clamp ON vs OFF: resolution **identical to the last digit** (d<sub>0</sub> 210.5 vs 210.0 &micro;m).
The clamp's only remaining job: keep the *persisted* &chi;&sup2; feature clean &mdash; with it off, the
tail re-explodes to ~10&#8313;&ndash;10&#185;&#8304; on ~0.1% of hits (a poisoned BDT input, not a crash).
Cleaner future fix: gate the persisted &chi;&sup2; instead of the angle.

## Backup: stage-4 per-flavor AUC (best seed)

| view / features | b | charm | light | gluon | &tau;&#8314; | &tau;&#8315; | &mu; | e |
|---|---|---|---|---|---|---|---|---|
| 1111 baseline | 0.706 | 0.569 | 0.590 | 0.748 | 0.706 | 0.727 | 0.911 | 0.879 |
| 1111 +refitBDT | 0.707 | 0.572 | 0.597 | 0.748 | **0.733** | 0.720 | 0.905 | 0.881 |
| 1100 baseline | 0.711 | 0.571 | 0.594 | 0.776 | 0.711 | 0.540 | 0.915 | 0.886 |
| 0000 baseline | 0.675 | 0.600 | 0.572 | 0.756 | 0.534 | 0.564 | 0.886 | 0.901 |
| 0000 +vtxDxy | 0.675 | 0.589 | 0.571 | 0.753 | 0.526 | **0.645** | 0.897 | 0.893 |

8-class one-vs-rest; ~1300 test jets/view; full 11-cell matrix in `eval_refitq/stage4/stage4_summary.json`.

## Backup: refit-BDT deployment path (parity receipts)

- Export: `train-refitquality --export-conifer` &rarr; conifer JSON + metadata (feature order, spec version, margin semantics, provenance)
- Margin semantics: conifer/producer compute the **raw booster logit margin**; the sklearn
  `predict(output_margin=True)` wrapper disagrees by up to 0.86 on xgboost&ge;2 &mdash; always reference the booster
- Producer parity: 16410/16455 trkMVA1 **bit-exact** vs the float32 offline walk; the 45 residuals are
  float32-snapshot values crossing strict-`<` split thresholds (one-leaf steps)
- Stage-3 v1 models: `conifer_margin_selfcheck = 0.0` on all 15 configs
- The score lands in `trkMVA1` on the refit track word &mdash; the hook the TS0 lever builds on